In [ ]:
import os
import nibabel as nib
from nilearn import plotting, datasets, image
import matplotlib.pyplot as plt
import numpy as np

# Load an image from a single brain
def img_load(folder, img):
    img_path = os.path.join(folder, img)
    if not os.path.exists(img_path):
        raise FileNotFoundError(f"File not found: {img_path}")
    fmri_image = nib.load(img_path)
    print(fmri_image.shape)
    return fmri_image

# Path to your image folder and file
folder = '/Users/X/PSILAUT/wDC/Output'
img = 'masked_within_A_drug_5mg_original-z-score-map_OVERLAP.nii.gz'

# Load the image
int_map = img_load(folder, img)

# Load the MNI152 template
mni_template = datasets.load_mni152_template()
# Load Yeo atlas for D
#yeo = datasets.fetch_atlas_yeo_2011()
#dmn_img = image.math_img("(img[..., 0] == 7).astype(float)", img=yeo.maps)

# Define the coordinates for multiple slices
cut_coords_axial = np.linspace(-40, 40, 7)
cut_coords_sagittal = np.linspace(-60, 20, 7)
cut_coords_coronal = np.linspace(-70, 30, 7)

# Create a plot of the image overlaid on the MNI template
fig, axes = plt.subplots(3, 7, figsize=(35, 15), facecolor='black')

# Determine min and max values for the colorbar
data = int_map.get_fdata()
vmin, vmax = np.min(data), np.max(data)
cmap = 'hot'

# Define colormap based on value range
#cmap = 'hot' if vmin >= 0 else 'cold_hot'

# Plot the statistical map for axial slices
for i, coord in enumerate(cut_coords_axial):
    display = plotting.plot_stat_map(        # ← added "display = "
        int_map, 
        bg_img=mni_template, 
        display_mode='z', 
        cut_coords=[coord], 
        colorbar=False, 
        axes=axes[0, i], 
        black_bg=True,
        annotate=False,
        cmap='hot',
        vmin=vmin,
        vmax=vmax,
        threshold=vmin
    )
    #display.add_contours(dmn_img, levels=[0.5], colors='cyan', linewidths=1.5)  # ← added

# Plot the statistical map for sagittal slices
for i, coord in enumerate(cut_coords_sagittal):
    display = plotting.plot_stat_map(        # ← added "display = "
        int_map, 
        bg_img=mni_template, 
        display_mode='x', 
        cut_coords=[coord], 
        colorbar=False, 
        axes=axes[1, i], 
        black_bg=True,
        annotate=False,
        cmap='hot',
        vmin=vmin,
        vmax=vmax,
        threshold=vmin
    )
    #display.add_contours(dmn_img, levels=[0.5], colors='cyan', linewidths=1.5)  # ← added

# Plot the statistical map for coronal slices
for i, coord in enumerate(cut_coords_coronal):
    display = plotting.plot_stat_map(        # ← added "display = "
        int_map, 
        bg_img=mni_template, 
        display_mode='y', 
        cut_coords=[coord], 
        colorbar=False, 
        axes=axes[2, i], 
        black_bg=True,
        annotate=False,
        cmap='hot',
        vmin=vmin,
        vmax=vmax,
        threshold=vmin
    )
    #display.add_contours(dmn_img, levels=[0.5], colors='cyan', linewidths=1.5)  # ← added

# Add a common colorbar
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.1, 0.02, 0.8])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)

# Customize colorbar ticks and labels
tick_fontsize = 12 * 1.75
label_fontsize = 14 * 1.75
cbar.set_ticks([vmin, vmax])
cbar.set_ticklabels([f'{vmin:.2f}', f'{vmax:.2f}'])
cbar.ax.tick_params(labelsize=tick_fontsize, colors='white')  # Increase font size for tick labels
cbar.ax.yaxis.set_tick_params(color='white')  # Set tick line color

# Add colorbar label with increased font size
cbar.set_label('Z-score', fontsize=label_fontsize, color='white')  # Increase font size for colorbar label

# Show the plot
plt.show()

# Save the figure as a PNG file with high resolution
fig.savefig('within_A_5mg.png', dpi=300, bbox_inches='tight')
